#Junyang You

# Take home exercise 3

In [0]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import tempfile
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

####I used several F1 datasets provided in Databricks. The main dataset is pit_stops, which records pit stop duration and related information. I also joined race data and race results to include more context, such as grid position and final position

In [0]:
df_pit = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True, inferSchema=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True, inferSchema=True)
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)

####I join the datasets to combine pit stop information with race level and performance level variables.

In [0]:
df = df_pit.join(df_races, "raceId", "left") \
           .join(df_results.select("raceId", "driverId", "grid", "positionOrder", "points"),
                 ["raceId", "driverId"], "left")

display(df)

####The target variable is pit stop duration (milliseconds). I select some features that affect pit stop performance, which includes stop number, lap number, race year and round, grid position, final race position, and points.

In [0]:
model_df= df.select(
    "milliseconds",
    "stop",
    "lap",
    "year",
    "round",
    "grid",
    "positionOrder",
    "points"
).dropna()

pdf = model_df.toPandas()
X = pdf.drop(columns=["milliseconds"])
y = pdf["milliseconds"]

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [0]:
username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
experiment_path = f"/Users/{username}/F1_MLflow_HW4"
mlflow.set_experiment(experiment_path)

In [0]:
def run_rf(run_name, params):
    with mlflow.start_run(run_name=run_name):
        rf = RandomForestRegressor(**params)
        rf.fit(X_train, y_train)
        preds = rf.predict(X_test)

        mlflow.sklearn.log_model(rf, "model")
        for k, v in params.items():
            mlflow.log_param(k, v)

        mse = mean_squared_error(y_test, preds)
        mae = mean_absolute_error(y_test, preds)
        r2 = r2_score(y_test, preds)

        mlflow.log_metric("mse", mse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)

        imp = pd.DataFrame({
            "feature": X_train.columns,
            "importance": rf.feature_importances_
        })

        tmp1 = tempfile.NamedTemporaryFile(delete=False, suffix=".csv")
        imp.to_csv(tmp1.name, index=False)
        mlflow.log_artifact(tmp1.name)

        residuals = y_test - preds
        plt.scatter(preds, residuals)
        plt.axhline(0)
        plt.title("Residual Plot")

        tmp2 = tempfile.NamedTemporaryFile(delete=False, suffix=".png")
        plt.savefig(tmp2.name)
        mlflow.log_artifact(tmp2.name)
        plt.close()

        return mse, r2

In [0]:
params_list = [
    {"n_estimators": 50, "max_depth": 5, "random_state": 42, "min_samples_split": 5, "min_samples_leaf": 2},
    {"n_estimators": 100, "max_depth": 5, "random_state": 42, "min_samples_split": 10, "min_samples_leaf": 2},
    {"n_estimators": 200, "max_depth": 5, "random_state": 42, "max_features": "sqrt",
    "bootstrap": True},
    {"n_estimators": 50, "max_depth": 10, "random_state": 42, "min_samples_split": 5, "min_samples_leaf": 2},
    {"n_estimators": 100, "max_depth": 10, "random_state": 42, "min_samples_split": 10, "min_samples_leaf": 5},
    {"n_estimators": 200, "max_depth": 10, "random_state": 42, "min_samples_split": 10,"min_samples_split": 10},
    {"n_estimators": 100, "max_depth": 10, "random_state": 42, "min_samples_leaf": 6,
    "max_features": "sqrt"},
    {"n_estimators": 200, "max_depth": 10, "random_state": 42, "n_jobs": -1, "min_samples_leaf": 2},
    {"n_estimators": 50, "max_depth": 15, "random_state": 42, "max_features": "sqrt", "min_samples_split": 10},
    {"n_estimators": 100, "max_depth": 15, "random_state": 42, "bootstrap": False},
    {"n_estimators": 200, "max_depth": 15, "random_state": 42, "n_jobs": -2},
    {"n_estimators": 300, "max_depth": 20, "random_state": 42, "min_samples_leaf": 4}
]


results = []

for i, p in enumerate(params_list):
    mse, r2 = run_rf(f"run_{i+1}", p)
    results.append({"run": i+1, "mse": mse, "r2": r2, **p})

In [0]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("mse")
results_df

####I selected model 11 to be my best model based on the lowest MSE because this is a regression problem. The best run achieved the lowest prediction error, meaning it predicted pit stop times more accurately than the other models. I also looked at R square to make sure the model explains some variation in the data. In addition, I checked the artifacts, including feature importance and residual plots, to make sure the model behavior looked reasonable instead of relying only on one metric.

In [0]:
from IPython.display import Image, display

display(Image("/Workspace/Users/jy3585@columbia.edu/take-home-exercise-3/example1.png"))

In [0]:
from IPython.display import Image, display

display(Image("/Workspace/Users/jy3585@columbia.edu/take-home-exercise-3/example2.png"))

In [0]:
from IPython.display import Image, display

display(Image("/Workspace/Users/jy3585@columbia.edu/take-home-exercise-3/homepage.png"))